# SuperBook Local Python Video Engine • Wan2.1 VACE 1.3B

This is the zero-cost GPU proof for the local Python backend. It does **not** modify the SuperBook Flutter app.

Pipeline: scene image → semantic motion prompt → Wan2.1 VACE 1.3B → MP4.

The same Python engine is also stored under `poc/python_video_engine/` so a future GPU machine can run it as an HTTP service.

In [ ]:
!nvidia-smi
!python -m pip install -q -U diffusers transformers accelerate safetensors pillow imageio[ffmpeg] fastapi uvicorn python-multipart

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('A Kaggle GPU is required for this proof. Turn Accelerator/GPU on.')

In [ ]:
from IPython.display import display
from PIL import Image
from kagglehub import model_download

# Upload the actual SuperBook scene image through the notebook file panel.
from IPython.display import Javascript
print('Upload one SuperBook scene image using the Kaggle notebook file panel, then set IMAGE_PATH below.')
IMAGE_PATH = '/kaggle/input/superbook-scene/scene.png'


In [ ]:
import os
from pathlib import Path
from PIL import Image

if not Path(IMAGE_PATH).exists():
    raise FileNotFoundError(f'Put your scene image at {IMAGE_PATH}, or edit IMAGE_PATH to the uploaded file path.')
image = Image.open(IMAGE_PATH).convert('RGB')
display(image.resize((min(768, image.width), int(image.height * min(768, image.width) / image.width))))

In [ ]:
import torch
from diffusers import DiffusionPipeline
from diffusers.utils import export_to_video

MODEL_ID = 'Wan-AI/Wan2.1-VACE-1.3B'
dtype = torch.float16

print('Loading', MODEL_ID)
pipe = DiffusionPipeline.from_pretrained(MODEL_ID, torch_dtype=dtype)
pipe.enable_model_cpu_offload()
print('Model loaded with CPU offload')

In [ ]:
MOTION_PROMPT = '''A cinematic storybook scene. The main character moves naturally through the existing scene: takes two slow steps toward the table, turns toward the other character, raises one hand while speaking, and naturally shifts body weight. Subtle breathing and clothing movement. The other character listens with small natural head and posture movement. Preserve the exact characters, faces, clothing, room, lighting, furniture and composition from the input image. No new characters, no scene change, no camera cut, no text, no morphing, no duplicated limbs.'''
print(MOTION_PROMPT)

In [ ]:
generator = torch.Generator(device='cuda').manual_seed(42)
result = pipe(
    image=image,
    prompt=MOTION_PROMPT,
    num_frames=49,
    num_inference_steps=20,
    generator=generator,
)
frames = result.frames[0]
OUT = '/kaggle/working/superbook_wan_vace.mp4'
export_to_video(frames, OUT, fps=16)
print('Generated:', OUT)

In [ ]:
from IPython.display import Video, display
display(Video('/kaggle/working/superbook_wan_vace.mp4', embed=True))

## Decision gate

PASS only if the clip shows meaningful character/body motion while preserving identity and scene continuity. Camera drift alone does not count. Major face/limb corruption does not count.

If it passes, the next step is to connect the already-isolated Python engine to the SuperBook Experience Player. If it fails, we keep the Python service contract and swap only the model backend.